In [ ]:
import pandas as pd
import torch
from transformers import pipeline
import tqdm
import time

In [ ]:
# Load the zero-shot classification pipeline
classifier1 = pipeline("zero-shot-classification", device=0)
classifier2 = pipeline("zero-shot-classification", model="roberta-large-mnli", device=0)
classifier3 = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)

In [ ]:
def biased_classification(texts, labels, bias_dict):
    # Perform zero-shot classification on all texts
    results = classifier1(texts, candidate_labels=labels, multi_label=False)
    
    # Prepare lists to store final results
    items = []
    final_labels = []
    final_scores = []
    
    # Adjust the scores for biased labels and determine the final label and score for each text
    for text, result in zip(texts, results):
        biased_scores = {label: score for label, score in zip(result['labels'], result['scores'])}
        for label, bias_factor in bias_dict.items():
            if label in biased_scores:
                biased_scores[label] += bias_factor
        final_label = max(biased_scores, key=biased_scores.get)
        final_score = biased_scores[final_label]
        
        # Append results to lists
        items.append(text)
        final_labels.append(final_label)
        final_scores.append(round(100*final_score)/100)
    
    # Create a DataFrame from the results
    df = pd.DataFrame({
        'item': items,
        'label': final_labels,
        'score': final_scores
    })
    
    return df

In [ ]:
%store -r items_ED5J990H5VAZT

In [ ]:
texts = items_ED5J990H5VAZT
labels = ['food', 'drink', 'merchandise']
bias_dict = {
    "food": 0.1,  # Adding a bias factor to 'food'
    "beverage": 0.1  # Adding a smaller bias factor to 'beverage'
}

before = time.time()

labels_ED5J990H5VAZT = biased_classification(texts, labels, bias_dict)

after = time.time()
print(after - before)

In [ ]:
%store labels_ED5J990H5VAZT